In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [9]:
#Read the original sales data
df = pd.read_csv('C:\\Users\\USER\\Documents\\Y3S1 works\\DWBI assignment 1\\original_sales_data.csv')

In [10]:
#display the colnums to verify the data
print(f"Columns : {list(df.columns)}")
print(f"Number of rows : {len(df)}")

Columns : ['Unnamed: 0', 'Sales_ID', 'Product_Category', 'Sales_Amount', 'Discount', 'Sales_Region', 'Date_of_Sale', 'Customer_Age', 'Customer_Gender', 'Sales_Representative']
Number of rows : 100000


In [11]:
print("\nFirst few rows:")
df.head()


First few rows:


,Unnamed: 0,Sales_ID,Product_Category,Sales_Amount,Discount,Sales_Region,Date_of_Sale,Customer_Age,Customer_Gender,Sales_Representative
0,0,efc35a5f-e48c-4803-9f6d-ef32a60e1367,Movies,341.05,45.98,East Dianeport,2024-07-12,69.0,Male,Bruce Rodriguez
1,1,92a86e42-de42-4e0d-89f3-bbb0b7f354bd,Pet Supplies,594.71,29.59,North Linda,2024-05-07,32.0,Other,Patricia Pugh
2,2,1fbbdc48-f405-48f3-8274-750581552d26,Outdoor,351.90,49.78,Lake Josephmouth,2024-05-11,28.0,Other,Kevin Fuentes
3,3,1f329e7a-13f0-4518-9eeb-84815301d64c,Electronics,NaN,NaN,North Martinside,2024-09-02,NaN,NaN,Evelyn Price
4,4,6ef559dc-3e68-4009-9845-4bab54d897c6,Books,795.75,27.21,Michealshire,2024-05-02,21.0,Other,Joseph Chavez


In [12]:
# Rename columns to standardized names for easier processing
column_mapping = {
    'Sales_ID': 'TransactionID',
    'Product_Category': 'ProductCategory',
    'Sales_Amount': 'SalesAmount',
    'Discount': 'Discount',
    'Sales_Region': 'Region',
    'Date_of_Sale': 'TransactionDate',
    'Customer_Age': 'CustomerAge',
    'Customer_Gender': 'CustomerGender',
    'Sales_Representative': 'SalesRep'
}

In [13]:
# Apply renaming 
df.rename(columns=column_mapping, inplace=True)

# Create Enriched Customer Data
Extract unique customers and add more attributes for a rich dimension

In [14]:
# Create customer dimension with enriched data
customers_df = df[['CustomerID']].drop_duplicates().copy() if 'CustomerID' in df.columns else pd.DataFrame()


In [15]:
# If no CustomerID exists, create one based on customer attributes
if customers_df.empty:
    # Create a composite key from customer attributes
    df['CustomerID'] = df.groupby(['CustomerAge', 'CustomerGender', 'Region']).ngroup() + 1000
    customers_df = df[['CustomerID', 'CustomerAge', 'CustomerGender', 'Region']].drop_duplicates().copy()


In [16]:
# Add enriched customer attributes
customers_df['CustomerName'] = [f'Customer_{i}' for i in range(1, len(customers_df) + 1)]

In [17]:
# Add customer segments based on age
def get_customer_segment(age):
    if age < 25:
        return 'Young Adult'
    elif age < 40:
        return 'Adult'
    elif age < 60:
        return 'Middle Age'
    else:
        return 'Senior'

customers_df['CustomerSegment'] = customers_df['CustomerAge'].apply(get_customer_segment)

In [18]:
# Add loyalty tier (random assignment for demonstration)
loyalty_tiers = ['Bronze', 'Silver', 'Gold', 'Platinum']
customers_df['LoyaltyTier'] = np.random.choice(loyalty_tiers, size=len(customers_df), p=[0.4, 0.3, 0.2, 0.1])

In [19]:
# Add customer tenure (random years between 1-10)
customers_df['CustomerTenureYears'] = np.random.randint(1, 11, size=len(customers_df))

In [20]:
print("\nSample customer data:")
print(customers_df.head())


Sample customer data:
   CustomerID  CustomerAge CustomerGender            Region CustomerName  \
0     87987.0         69.0           Male    East Dianeport   Customer_1   
1     26249.0         32.0          Other       North Linda   Customer_2   
2     19317.0         28.0          Other  Lake Josephmouth   Customer_3   
3         NaN          NaN            NaN  North Martinside   Customer_4   
4      7293.0         21.0          Other      Michealshire   Customer_5   

  CustomerSegment LoyaltyTier  CustomerTenureYears  
0          Senior      Silver                    8  
1           Adult      Bronze                    6  
2           Adult      Silver                    4  
3          Senior      Bronze                    5  
4     Young Adult        Gold                    4  


# Create Enriched Product Data
Extract product categories and create product dimension

In [21]:
# Create product dimension from categories
unique_categories = df['ProductCategory'].unique()
products_df = pd.DataFrame({
    'ProductID': [f'PROD_{i:03d}' for i in range(1, len(unique_categories) + 1)],
    'ProductCategory': unique_categories,
})

In [22]:
# Add product hierarchy (Category -> SubCategory -> ProductName)
category_hierarchy = {
    'Electronics': {'SubCategory': ['Smartphones', 'Laptops', 'Accessories'], 
                    'Products': ['iPhone 13', 'MacBook Pro', 'AirPods', 'Samsung TV', 'Wireless Mouse']},
    'Clothing': {'SubCategory': ['Men', 'Women', 'Kids'], 
                 'Products': ['Jeans', 'T-Shirts', 'Dresses', 'Jackets', 'Shoes']},
    'Home & Garden': {'SubCategory': ['Furniture', 'Decor', 'Kitchen'], 
                      'Products': ['Sofa', 'Table Lamp', 'Cookware Set', 'Bed Sheets', 'Garden Tools']},
    'Sports': {'SubCategory': ['Fitness', 'Outdoor', 'Team Sports'], 
               'Products': ['Yoga Mat', 'Tent', 'Soccer Ball', 'Dumbbells', 'Bicycle']},
    'Books': {'SubCategory': ['Fiction', 'Non-Fiction', 'Educational'], 
              'Products': ['Novel', 'Biography', 'Textbook', 'Children Book', 'Cookbook']}
}

In [23]:
# Assign subcategories and product names based on category
def enrich_product_data(row):
    category = row['ProductCategory']
    if category in category_hierarchy:
        subcats = category_hierarchy[category]['SubCategory']
        products = category_hierarchy[category]['Products']
        row['SubCategory'] = random.choice(subcats)
        row['ProductName'] = random.choice(products)
    else:
        row['SubCategory'] = 'General'
        row['ProductName'] = f'{category} Item'
    return row

products_df = products_df.apply(enrich_product_data, axis=1)

In [24]:
# Add product attributes
products_df['UnitPrice'] = np.random.uniform(10, 500, size=len(products_df)).round(2)
products_df['Supplier'] = [f'Supplier_{chr(65+i)}' for i in range(len(products_df))]

In [25]:
print("\nSample product data:")
print(products_df.head())


Sample product data:
  ProductID ProductCategory  SubCategory        ProductName  UnitPrice  \
0  PROD_001          Movies      General        Movies Item     356.65   
1  PROD_002    Pet Supplies      General  Pet Supplies Item     341.10   
2  PROD_003         Outdoor      General       Outdoor Item     383.85   
3  PROD_004     Electronics  Accessories            AirPods      63.46   
4  PROD_005           Books  Non-Fiction          Biography     105.67   

     Supplier  
0  Supplier_A  
1  Supplier_B  
2  Supplier_C  
3  Supplier_D  
4  Supplier_E  


# Create Sales Representative Dimension

In [26]:
# Extract unique sales representatives
unique_reps = df['SalesRep'].unique()
reps_df = pd.DataFrame({
    'SalesRepID': [f'REP_{i:03d}' for i in range(1, len(unique_reps) + 1)],
    'SalesRepName': unique_reps,
})

In [27]:
# Add representative attributes
rep_regions = ['North', 'South', 'East', 'West', 'Central']
rep_teams = ['Alpha', 'Beta', 'Gamma', 'Delta']
rep_titles = ['Junior Rep', 'Senior Rep', 'Lead Rep', 'Manager']
reps_df['Region'] = np.random.choice(rep_regions, size=len(reps_df))
reps_df['Team'] = np.random.choice(rep_teams, size=len(reps_df))
reps_df['Title'] = np.random.choice(rep_titles, size=len(reps_df))
reps_df['HireDate'] = [datetime.now() - timedelta(days=random.randint(365, 1825)) for _ in range(len(reps_df))]
reps_df['HireDate'] = reps_df['HireDate'].dt.strftime('%Y-%m-%d')

In [28]:
print("\nSample representative data:")
print(reps_df.head())


Sample representative data:
  SalesRepID     SalesRepName   Region   Team       Title    HireDate
0    REP_001  Bruce Rodriguez  Central  Gamma  Junior Rep  2024-11-26
1    REP_002    Patricia Pugh     West  Gamma  Junior Rep  2021-07-23
2    REP_003    Kevin Fuentes     West  Delta     Manager  2023-08-03
3    REP_004     Evelyn Price     West  Gamma    Lead Rep  2022-02-15
4    REP_005    Joseph Chavez     West   Beta    Lead Rep  2023-01-18


# Create Transaction Facts (for SQL Server)

In [29]:
# Create the core transaction fact table
transactions_df = df.copy()

In [30]:
# Add derived columns
transactions_df['NetAmount'] = transactions_df['SalesAmount'] - transactions_df['Discount']
transactions_df['TransactionHour'] = pd.to_datetime(transactions_df['TransactionDate']).dt.hour

In [31]:
# Add foreign keys to dimensions
# Map ProductCategory to ProductID
category_to_prodid = dict(zip(products_df['ProductCategory'], products_df['ProductID']))
transactions_df['ProductID'] = transactions_df['ProductCategory'].map(category_to_prodid)

In [32]:
# Map SalesRep to SalesRepID
rep_to_repid = dict(zip(reps_df['SalesRepName'], reps_df['SalesRepID']))
transactions_df['SalesRepID'] = transactions_df['SalesRep'].map(rep_to_repid)

In [ ]:
# Map SalesRep to SalesRepID
rep_to_repid = dict(zip(reps_df['SalesRepName'], reps_df['SalesRepID']))
transactions_df['SalesRepID'] = transactions_df['SalesRep'].map(rep_to_repid

In [33]:
# Keep only relevant columns for the fact table
fact_columns = ['TransactionID', 'TransactionDate', 'ProductID', 'CustomerName', 
                'SalesRepID', 'SalesAmount', 'Discount', 'NetAmount']
transactions_df = transactions_df[fact_columns]

KeyError: "['CustomerName'] not in index"

In [ ]:
print("\nSample transaction data:")
print(transactions_df.head())


Sample transaction data:
                          TransactionID TransactionDate ProductID  CustomerID  \
0  efc35a5f-e48c-4803-9f6d-ef32a60e1367      2024-07-12  PROD_001     87987.0   
1  92a86e42-de42-4e0d-89f3-bbb0b7f354bd      2024-05-07  PROD_002     26249.0   
2  1fbbdc48-f405-48f3-8274-750581552d26      2024-05-11  PROD_003     19317.0   
3  1f329e7a-13f0-4518-9eeb-84815301d64c      2024-09-02  PROD_004         NaN   
4  6ef559dc-3e68-4009-9845-4bab54d897c6      2024-05-02  PROD_005      7293.0   

  SalesRepID  SalesAmount  Discount  NetAmount  
0    REP_001       341.05     45.98     295.07  
1    REP_002       594.71     29.59     565.12  
2    REP_003       351.90     49.78     302.12  
3    REP_004          NaN       NaN        NaN  
4    REP_005       795.75     27.21     768.54  


# Export Data to Multiple Source Formats

In [ ]:
# Export Customer Dimension to CSV
customers_df.to_csv(f'customers.csv', index=False)
print("Exported customers.csv")

Exported customers.csv


In [ ]:
# Export Product Dimension to CSV
products_df.to_csv(f'products.csv', index=False)
print("Exported products.csv")

Exported products.csv


In [ ]:
# Export Sales Representative Dimension to CSV
reps_df.to_csv(f'sales_reps.csv', index=False)
print("Exported sales_reps.csv")

Exported sales_reps.csv


In [ ]:
# Export Transactions to CSV (this will be loaded into SQL Server)
transactions_df.to_csv(f'transactions.csv', index=False)
print(" Exported transactions.csv")

 Exported transactions.csv
